In [0]:
%sql
-- Source table (simulates CDC events from a system)
CREATE OR REPLACE TABLE workspace.default.customer_changes_raw (
    customer_id INT,
    name STRING,
    city STRING,
    operation STRING,
    updated_at TIMESTAMP
);

INSERT INTO workspace.default.customer_changes_raw VALUES
    (1, 'Alice', 'NYC',    'INSERT', '2024-01-01 10:00:00'),
    (1, 'Alice', 'LA',     'UPDATE', '2024-06-15 10:00:00'),
    (1, 'Alice', 'SF',     'UPDATE', '2024-10-01 10:00:00'),
    (2, 'Bob',   'Boston', 'INSERT', '2024-02-01 10:00:00'),
    (2, 'Bob',   'Chicago','UPDATE', '2024-08-01 10:00:00'),
    (3, 'Carol', 'Miami',  'INSERT', '2024-03-01 10:00:00');

SELECT * FROM workspace.default.customer_changes_raw ORDER BY customer_id, updated_at;

In [0]:
%sql
-- Bronze: raw copy of the CDC events
CREATE OR REPLACE TABLE workspace.default.bronze_customer_changes AS
SELECT
    *,
    current_timestamp() AS _ingested_at
FROM workspace.default.customer_changes_raw;

SELECT * FROM workspace.default.bronze_customer_changes;

`Why Bronze? A frozen copy of source data. If we need to reprocess, we use this instead of re-querying the source.`

In [0]:

%sql
--Silver Layer — Create the SCD Type 2 Table Structure
-- Silver: empty SCD Type 2 table
CREATE OR REPLACE TABLE workspace.default.silver_dim_customer (
    customer_id INT,
    name STRING,
    city STRING,
    __start_date DATE,       -- YOU define this
    __end_date   DATE,       -- YOU define this
    __is_current BOOLEAN     -- YOU define this
);

SELECT * FROM workspace.default.silver_dim_customer;  -- empty




In [0]:
%sql
--Load the FIRST Version of Each Customer (We need to seed the SCD table with the earliest event for each customer.)
INSERT INTO workspace.default.silver_dim_customer
SELECT
    customer_id,
    name,
    city,
    CAST(updated_at AS DATE) AS __start_date,
    DATE('9999-12-31')       AS __end_date,
    true                     AS __is_current
FROM (
    -- Get the FIRST event per customer
    SELECT
        customer_id,
        name,
        city,
        updated_at,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at ASC) AS rn
    FROM workspace.default.bronze_customer_changes
) t
WHERE rn = 1;

SELECT * FROM workspace.default.silver_dim_customer ORDER BY customer_id;

In [0]:
%sql
--Create the Updates View (Now we need a table of later changes (not the first event) to process as SCD updates:)
CREATE OR REPLACE TABLE workspace.default.updates AS
SELECT
    customer_id,
    name,
    city,
    updated_at
FROM (
    SELECT
        customer_id, name, city, updated_at,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at ASC) AS rn
    FROM workspace.default.bronze_customer_changes
) t
WHERE rn > 1;   -- Skip the first event (already inserted)

SELECT * FROM workspace.default.updates ORDER BY customer_id, updated_at;

In [0]:
%sql
--SCD Type 2 — Step 1 (Close Old Versions)
MERGE INTO workspace.default.silver_dim_customer AS t
USING workspace.default.updates AS s
ON t.customer_id = s.customer_id AND t.__is_current = true
WHEN MATCHED AND (t.name != s.name OR t.city != s.city) THEN
    UPDATE SET
        t.__is_current = false,
        t.__end_date   = CAST(s.updated_at AS DATE);

SELECT * FROM workspace.default.silver_dim_customer ORDER BY customer_id        

`updates has two rows for Alice (LA and SF). Both match the same target row (Alice-NYC, current), and Delta MERGE doesn’t allow that:
[DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE]`

In [0]:
%sql
MERGE INTO workspace.default.silver_dim_customer AS t
USING (
    -- Only the LATEST update per customer
    SELECT customer_id, name, city, updated_at
    FROM (
        SELECT
            customer_id, name, city, updated_at,
            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
        FROM workspace.default.updates
    ) x
    WHERE rn = 1
) AS s
ON t.customer_id = s.customer_id AND t.__is_current = true
WHEN MATCHED AND (t.name != s.name OR t.city != s.city) THEN
    UPDATE SET
        t.__is_current = false,
        t.__end_date   = CAST(s.updated_at AS DATE);

In [0]:
%sql
--SCD Type 2 — Step 2 (Insert New Versions)
INSERT INTO workspace.default.silver_dim_customer
SELECT
    s.customer_id,
    s.name,
    s.city,
    CAST(s.updated_at AS DATE) AS __start_date,
    DATE('9999-12-31')         AS __end_date,
    true                       AS __is_current
FROM (
    SELECT customer_id, name, city, updated_at
    FROM (
        SELECT
            customer_id, name, city, updated_at,
            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
        FROM workspace.default.updates
    ) x
    WHERE rn = 1
) s
LEFT JOIN workspace.default.silver_dim_customer t
    ON t.customer_id = s.customer_id AND t.__is_current = true
WHERE t.customer_id IS NULL
   OR t.name != s.name
   OR t.city != s.city;

In [0]:
%sql
SELECT
    customer_id, name, city, __start_date, __end_date, __is_current
FROM workspace.default.silver_dim_customer
ORDER BY customer_id, __start_date;

In [0]:
%sql
--Gold Layer — Aggregation
CREATE OR REPLACE TABLE workspace.default.gold_customer_movement AS
SELECT
    city,
    COUNT(DISTINCT customer_id) AS customers_in_city,
    MIN(__start_date) AS first_seen,
    MAX(__start_date) AS last_updated
FROM workspace.default.silver_dim_customer
GROUP BY city;

SELECT * FROM workspace.default.gold_customer_movement ORDER BY customers_in_city DESC;

**The Killer Query — Time-Based Lookup**
**"Where was Alice on 2024-08-01?"**

`EXPECTED :Alice in LA. `
    ```Why: Her LA period ran from 2024-06-15 to 2024-10-01.```

    ```This is the SCD Type 2 superpower — the time-based join.```

In [0]:
%sql
SELECT customer_id, name, city
FROM workspace.default.silver_dim_customer
WHERE customer_id = 1
  AND '2024-08-01' BETWEEN __start_date AND __end_date;